# Signature-PDE Kernel

Approximates the *untruncated* signature kernel by solving a Goursat PDE ([Salvi et al., 2021](https://arxiv.org/pdf/2006.14794.pdf)). No truncation level — `difference` toggles taking increments of the lifted paths.

## Environment
Detect the live backend/device and whether a SYCL fast-path is available.

In [ ]:
import sys, pathlib
# Make `_nbtools` and the in-repo `ksig` importable whether the notebook is
# launched from ./notebooks or from the repo root (no `pip install` needed).
_nbdir = pathlib.Path.cwd()
_root = _nbdir.parent if (_nbdir / "_nbtools.py").exists() else _nbdir
_nbdir = _root / "notebooks"
for _p in (str(_nbdir), str(_root)):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import numpy as np
import ksig
import _nbtools as nb
%matplotlib inline

ENV = nb.detect_env()
nb.print_env_banner(ENV)

## Deterministic input
`simulate(16, 20, 3, seed=0)` — a batch of integrated random walks (portable: the NumPy RNG is bit-identical on every machine, so the values below are reproducible on Aurora).

In [ ]:
X = nb.simulate(16, 20, 3, seed=0)
print('X shape:', X.shape, '| dtype:', X.dtype)

## Compute the kernel

```python
pde = ksig.kernels.SignaturePDEKernel(difference=True, normalize=True, static_kernel=ksig.static.kernels.RBFKernel())
K = pde(X)
print("gram shape :", tuple(K.shape))
print("K[:3,:3]   :\n", np.round(nb.as_host(K)[:3, :3], 4))
print("diag mean  :", round(float(np.diag(nb.as_host(K)).mean()), 6))
```

**Reference output — NVIDIA H100 NVL (CuPy 14.1.0), kwargs: `difference=True, normalize=True, static=RBF`:**

```text
gram shape : (16, 16)
K[:3,:3]   :
 [[1.     0.2979 0.4118]
 [0.2979 1.     0.3314]
 [0.4118 0.3314 1.    ]]
diag mean  : 1.0
```

Running the next cell on the target machine should reproduce this to ~1e-8 in
float64 (`diag == 1` exactly when `normalize=True`).

In [ ]:
pde = ksig.kernels.SignaturePDEKernel(difference=True, normalize=True, static_kernel=ksig.static.kernels.RBFKernel())
K = pde(X)
print("gram shape :", tuple(K.shape))
print("K[:3,:3]   :\n", np.round(nb.as_host(K)[:3, :3], 4))
print("diag mean  :", round(float(np.diag(nb.as_host(K)).mean()), 6))

## Scaling — green = CUDA reference, blue = this machine

The cell below sweeps **sequence length L  (n=32, d=3)** and times each point on whatever backend
is live, then overlays:

* 🟩 **green** — the frozen reference measured on **NVIDIA H100 NVL** (`cuda_reference.json`),
* 🟦 **blue** — what *this* machine computes now (torch-native on Aurora XPU / CUDA / CPU),
* 🟧 **orange** — the optional **SYCL** fast-path, drawn **only if** a `ksig._sycl`
  build is present (`nb.sycl_available()`); if SYCL is never adopted, no orange curve.

The grid and the knobs at the top of the cell are **tunable** — they default to
the reference grid so blue lines up with green; widen them to push the frontier.

In [ ]:
# --- tunable knobs (default to the CUDA-reference grid) ---------------------
L_GRID = [8, 16, 32, 64, 128]          # sequence lengths to sweep
N, D   = 32, 3                 # fixed batch size / channels
REPS   = 5
DIFFERENCE = True                 # increments of the lifted path

def time_one(L):
    Xs = nb.simulate(N, L, D, seed=1)
    k = ksig.kernels.SignaturePDEKernel(difference=DIFFERENCE, normalize=True, static_kernel=ksig.static.kernels.RBFKernel())
    return nb.timeit(lambda: k(Xs), reps=REPS, device=ENV["device"])

times = [time_one(L) for L in L_GRID]

# Second (orange) curve ONLY when a SYCL build is available.
sycl_times = None
if nb.sycl_available():
    nb.enable_sycl(True)
    sycl_times = [time_one(L) for L in L_GRID]
    nb.enable_sycl(False)

nb.scaling_plot(L_GRID, times, "signature_pde_kernel", sycl_seconds=sycl_times,
                title="signature_pde_kernel — wall time vs sequence length");